<a href="https://colab.research.google.com/github/pejmanrasti/SMART_MICROSCOPY/blob/main/AgenticAI/01_RAG_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade langchain langchain-community langchain-text-splitters langchain-openai chromadb pypdf sentence-transformers langchain-huggingface

# Automatically restart the Python kernel to clear Colab's import package cache
import os
os.kill(os.getpid(), 9)

### 1. Setup and Authentication
First, we import the necessary libraries and set up the OpenRouter API key. Ensure you have added your `OPENROUTER_API_KEY` to the Colab Secrets panel on the left.

In [ ]:
import os
from google.colab import userdata

# Direct imports from their respective modular packages
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# We bypass the 'langchain.chains' module completely by importing core components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Securely load the API key
try:
    OPENROUTER_API_KEY = userdata.get('Your_API_Key')
except Exception:
    OPENROUTER_API_KEY = "Alternative_API_KEY"  # Fallback

# Initialize the LLM via OpenRouter
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    model="nvidia/nemotron-3-ultra-550b-a55b:free"
)

# Initialize free HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

### 2. Load and Process Documents
Create a folder named `papers` in your Colab files panel and upload your PDF papers there. This code will load them, extract the text, and split them into smaller chunks.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os


# 2. Define the path to your zip file on Google Drive
# Update this path to where your zip file is located on your Drive
zip_path = '/content/drive/MyDrive/01_Aussois_formation/Papers.zip'
extract_to = '/content/papers'

# 3. Unzip the file if it exists
if os.path.exists(zip_path):
    print(f"Extracting {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction complete! Use '/content/papers' in the next step.")
else:
    print(f"Zip file not found at: {zip_path}. Please update the path and try again.")

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader


documents = []

# Scan the directory and safely load individual PDF files
if os.path.exists("papers"):
    pdf_files = [f for f in os.listdir("papers/Papers") if f.endswith(".pdf")]

    if not pdf_files:
        print("No PDF files found in the 'papers/' directory. Please upload your papers.")

    for file_name in pdf_files:
        file_path = os.path.join("papers/Papers", file_name)
        try:
            loader = PyPDFLoader(file_path)
            loaded_docs = loader.load()
            documents.extend(loaded_docs)
            print(f"Successfully loaded: {file_name} ({len(loaded_docs)} pages)")
        except Exception as e:
            print(f"Warning: Skipping file '{file_name}' due to parsing error. The file might be corrupted, empty, or still uploading. (Error: {e})")

if documents:
    print(f"\nTotal loaded pages: {len(documents)}. Splitting text...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} text chunks.")
else:
    chunks = []
    print("No valid documents were successfully loaded. Please make sure your PDFs are completely uploaded and healthy.")

### 3. Create Vector Store and RAG Chain
Now we store the chunks in a Chroma vector database and link it to our LLM to create the chat chain.

In [ ]:
if chunks:
    print("Creating vector database (this might take a moment)......\n")
    vectorstore = Chroma.from_documents(chunks, embeddings)

    # Increased 'k' from 4 to 10 so the system retrieves more context chunks
    # to find detailed benefits across your 152 pages of papers!
    retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

    # Helper function to format retrieved documents with their filenames/sources
    def format_docs_with_sources(docs):
        formatted_segments = []
        for doc in docs:
            # Try to get the filename from metadata
            source_path = doc.metadata.get('source', 'Unknown Source')
            filename = os.path.basename(source_path)
            segment = f"[Source File: {filename}]\n{doc.page_content}"
            formatted_segments.append(segment)
        return "\n\n---\n\n".join(formatted_segments)

    # Define the system prompt with strict instruction to cite source filenames
    system_prompt = (
        "You are an assistant for question-answering tasks for a workshop.\n\n"
        "Use the following pieces of retrieved context to answer the question. "
        "In your answer, you MUST explicitly cite which source files (e.g., '[Source File: s42003-023-04857-4.pdf]') "
        "your claims/points came from. Only cite the files present in the context below.\n\n"
        "If you don't know the answer based on the context, say that you don't know.\n\n"
        "Context:\n{context}\n\n"
        "Question: {input}\n"
        "Answer:"
    )

    prompt = ChatPromptTemplate.from_template(system_prompt)

    # We create a dictionary output chain to easily access both the response and the original documents
    rag_chain = (
        {
            "context": retriever | format_docs_with_sources,
            "input": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    # We also keep a helper pipeline to inspect raw retrieved source docs
    retrieval_chain = retriever

    print("\nRAG System with Citations (Expanded Search k=10) is ready!")
else:
    print("No documents found! Please upload PDFs to the 'papers' folder and re-run the previous cell.")

### 4. Chat with your Papers
Run this cell to ask questions about the papers you uploaded.

In [ ]:
question = "What are the main benefits of smart microscopy discussed in the papers?"

if chunks:
    # 1. Fetch response from our citation-aware chain
    response = rag_chain.invoke(question)

    # 2. Retrieve the raw source documents used
    retrieved_docs = retrieval_chain.invoke(question)

    print("Question:", question)
    print("\nAnswer with Citations:")
    print(response)

    # 3. Print the source document summary
    print("\n" + "="*40)
    print("Source Documents Retrieved:")
    seen_sources = set()
    for doc in retrieved_docs:
        source_path = doc.metadata.get('source', 'Unknown Source')
        filename = os.path.basename(source_path)
        if filename not in seen_sources:
            print(f"- {filename} (Page {doc.metadata.get('page', 0) + 1})")
            seen_sources.add(filename)

### 🛠️ Hands-on Workshop Challenges (For All Participants!)
No programming experience required! Below are 4 interactive tasks to help you explore how a Retrieval-Augmented Generation (RAG) system thinks, how it retrieves information, and how to improve its performance.

---

#### 📂 Challenge 1: Bring Your Own Paper (Data Upload)
Let's see how the RAG system handles completely new information.
1. Go to the **Files panel** on the left side of Colab (the 📂 folder icon).
2. Open the `papers` folder.
3. Drag and drop any scientific PDF of your choice from your computer into that folder.
4. Go back up and run **Step 2 (Load and Process Documents)** to process your new file.
5. Run **Step 3** to index it, and ask a question in **Step 4** about your specific paper to see if it correctly cites your newly uploaded file!

---

#### 🎛️ Challenge 2: Adjust the "Search Sensitivity" (Parameter Tuning)
How much context does an AI need to answer a question properly?
1. In **Step 3 (Create Vector Store)**, find this line of code:  
   `retriever = vectorstore.as_retriever(search_kwargs={"k": 10})`
2. Change `10` to `2` (retrieving very little context) and run the cell. Then run the final question cell. What happens to the quality of the answer?
3. Now change it to `15` (retrieving a large amount of context) and run it again. Does the answer become more detailed, or does it become too cluttered?

---

#### 🔍 Challenge 3: Play "AI Detective" (Citation Verification)
Large Language Models can sometimes "hallucinate" (make up information) or cite the wrong sources.
1. Ask the system a highly specific question in **Step 4**.
2. Look at the generated answer and the source citation (e.g., `[Source File: s42003-023-04857-4.pdf]`).
3. Look at the **Source Documents Retrieved** summary printed at the very bottom of the cell (which shows exactly what pages were extracted).
4. Double-check: Did the AI actually answer using facts from those specific pages, or did it generate something unrelated? Discuss your findings with your team!

---

#### ✍️ Challenge 4: Prompt Engineering (Modify the Persona)
How we ask a model to behave changes the structure of its answers.
1. Go to **Step 3** and look at the `system_prompt` variable.
2. Try changing the instructions inside the text. For example, add:
   * *"Explain your answer as if you are teaching a 10-year-old child..."*
   * *"Structure your answer strictly as a bulleted executive summary of exactly 3 points..."*
   * *"Respond in French, but keep the source citations in English..."*
3. Re-run **Step 3** and **Step 4** to see how the formatting of your answers adapts to your new instructions!